# FOA-Attack on Kaggle — `code_foa_attack_v1.0`

Notebook này được chuẩn bị cho:

- **Repo:** `jiaxiaojunQAQ/FOA-Attack`
- **Commit cố định:** `6aa9ad463a1e812b7f77ff2e5c4b1712e7c78c4b`
- **Script attack:** `generate_adversarial_samples_foa_attack.py`
- **Dataset Kaggle:** `/kaggle/input/blackbox-adversarial-attack-dataset`

## Khác biệt quan trọng với Kaggle

1. Repo pin `transformers==4.49.0`; Kaggle image mới có thể dùng Transformers 5.x. Notebook tạo **venv riêng** và pin 4.49.0.
2. `Base.py` import `kmeans_pytorch`, nhưng package này bị thiếu trong `requirements.txt`. Notebook thêm **`kmeans-pytorch==0.3`**.
3. `config_schema.py` dùng dataclass instance làm default. Python 3.12 có thể báo lỗi mutable default. Notebook tự patch sang `field(default_factory=...)`.
4. `FOAttack.py` hard-code `CUDA_VISIBLE_DEVICES="2"`. Notebook bỏ hard-code này để Kaggle dùng `cuda:0`.
5. Không `pip install -r requirements.txt` nguyên xi vì file đó pin cả CUDA/Torch/Jupyter và rất nhiều package không cần cho bước sinh adversarial image.

## Dependency tối thiểu cho attack generation

| Import | Package | Version repo | Notebook |
|---|---|---:|---:|
| `torch` | torch | 2.6.0 | giữ Torch GPU của Kaggle; có tùy chọn pin 2.6.0 |
| `torchvision` | torchvision | 0.21.0 | giữ pair tương thích với Torch của Kaggle |
| `transformers` | transformers | 4.49.0 | **4.49.0** |
| `hydra` | hydra-core | 1.3.2 | **1.3.2** |
| `omegaconf` | omegaconf | 2.3.0 | **2.3.0** |
| `pytorch_lightning` | pytorch-lightning | 2.5.0.post0 | **2.5.0.post0** |
| `wandb` | wandb | 0.19.1 | **0.19.1**, disabled mode |
| `kmeans_pytorch` | kmeans-pytorch | thiếu trong requirements | **0.3** |
| `yaml` | PyYAML | 6.0.2 | **6.0.2** |
| `ftfy` | ftfy | 6.3.1 | **6.3.1** |
| `tqdm` | tqdm | 4.67.1 | **4.67.1** |
| `numpy` | numpy | 1.26.4 | giữ NumPy của Kaggle để tránh binary conflict |
| `PIL` | Pillow | 11.1.0 | giữ Pillow của Kaggle |

> Ba surrogate model được tải từ Hugging Face. Nếu chưa có cache/model trong dataset, cần bật **Internet** ở Kaggle.


In [ ]:
# Cell 1 — User parameters

from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/blackbox-adversarial-attack-dataset")
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "FOA-Attack"
VENV_DIR = WORK_ROOT / "foa_venv"
HF_HOME = WORK_ROOT / "hf_cache"
OUTPUT_DIR = WORK_ROOT / "foa_attack_output"

REPO_URL = "https://github.com/jiaxiaojunQAQ/FOA-Attack.git"
REPO_COMMIT = "6aa9ad463a1e812b7f77ff2e5c4b1712e7c78c4b"

SMOKE_NUM_SAMPLES = 1
SMOKE_STEPS = 2

PAPER_NUM_SAMPLES = 100
PAPER_STEPS = 300

# False = reuse Kaggle's CUDA-enabled torch/torchvision pair.
# True = install author's exact torch==2.6.0 / torchvision==0.21.0 inside the venv.
FORCE_PAPER_TORCH = False

assert DATASET_ROOT.exists(), (
    f"Dataset not found at {DATASET_ROOT}. "
    "Attach zintom/blackbox-adversarial-attack-dataset first."
)

print("DATASET_ROOT:", DATASET_ROOT)
print("REPO_DIR:", REPO_DIR)
print("VENV_DIR:", VENV_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# Cell 2 — Inspect Kaggle runtime without importing ML libraries.

import sys
import platform
import subprocess
from importlib import metadata

packages = [
    "torch", "torchvision", "transformers", "hydra-core", "omegaconf",
    "pytorch-lightning", "wandb", "numpy", "Pillow", "tqdm"
]

print("Python:", sys.version)
print("Platform:", platform.platform())

for pkg in packages:
    try:
        print(f"{pkg:20s} {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:20s} <not installed>")

print("\nGPU:")
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# Cell 3 — Create an isolated Kaggle-safe environment.

import subprocess
import sys

if not VENV_DIR.exists():
    subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)],
        check=True,
    )

VENV_PY = VENV_DIR / "bin" / "python"

subprocess.run(
    [str(VENV_PY), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)

minimal_packages = [
    "transformers==4.49.0",
    "tokenizers==0.21.0",
    "huggingface-hub==0.28.1",
    "safetensors==0.5.0",
    "hydra-core==1.3.2",
    "omegaconf==2.3.0",
    "pytorch-lightning==2.5.0.post0",
    "torchmetrics==1.6.1",
    "lightning-utilities==0.11.9",
    "wandb==0.19.1",
    "kmeans-pytorch==0.3",
    "PyYAML==6.0.2",
    "ftfy==6.3.1",
    "tqdm==4.67.1",
]

subprocess.run(
    [str(VENV_PY), "-m", "pip", "install", "-q", "--no-cache-dir", *minimal_packages],
    check=True,
)

if FORCE_PAPER_TORCH:
    subprocess.run(
        [
            str(VENV_PY), "-m", "pip", "install", "-q", "--no-cache-dir",
            "--index-url", "https://download.pytorch.org/whl/cu124",
            "torch==2.6.0", "torchvision==0.21.0",
        ],
        check=True,
    )

print("Environment ready:", VENV_PY)


In [ ]:
# Cell 4 — Get FOA-Attack source code.
# Prefer a copy already stored in the attached dataset; otherwise clone GitHub.

import shutil
import subprocess

def find_repo_in_dataset(root):
    hits = []
    for script in root.rglob("generate_adversarial_samples_foa_attack.py"):
        parent = script.parent
        if (parent / "surrogates").exists() and (parent / "config").exists():
            hits.append(parent)
    return hits

repo_candidates = find_repo_in_dataset(DATASET_ROOT)
print("Repo candidates inside dataset:", repo_candidates)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if repo_candidates:
    shutil.copytree(repo_candidates[0], REPO_DIR)
    print("Copied repo from:", repo_candidates[0])
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_COMMIT], check=True)
    print("Cloned commit:", REPO_COMMIT)

assert (REPO_DIR / "generate_adversarial_samples_foa_attack.py").exists()
assert (REPO_DIR / "surrogates").exists()
assert (REPO_DIR / "config").exists()

print("FOA repo ready:", REPO_DIR)


In [ ]:
# Cell 5 — Patch Python 3.12/Kaggle incompatibilities.
# These changes do not alter FOA loss, perturbation update, crop policy, or attack hyperparameters.

import subprocess

schema_path = REPO_DIR / "config_schema.py"

patched_schema = """from dataclasses import dataclass, field
from typing import Optional
from hydra.core.config_store import ConfigStore


@dataclass
class WandbConfig:
    entity: str = "???"
    project: str = "local_adversarial_attack"


@dataclass
class BlackboxConfig:
    model_name: str = "gpt4v"
    batch_size: int = 1
    timeout: int = 30


@dataclass
class DataConfig:
    batch_size: int = 1
    num_samples: int = 1000
    cle_data_path: str = "resources/images/bigscale"
    tgt_data_path: str = "resources/images/target_images"
    output: str = "./Ours"


@dataclass
class OptimConfig:
    alpha: float = 1.0
    epsilon: int = 8
    steps: int = 300


@dataclass
class ModelConfig:
    input_res: int = 336
    use_source_crop: bool = True
    use_target_crop: bool = True
    crop_scale: tuple = (0.5, 0.9)
    ensemble: bool = True
    device: str = "cuda:0"
    backbone: list = ("L336", "B16", "B32", "Laion")


@dataclass
class MainConfig:
    data: DataConfig = field(default_factory=DataConfig)
    optim: OptimConfig = field(default_factory=OptimConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    wandb: WandbConfig = field(default_factory=WandbConfig)
    blackbox: BlackboxConfig = field(default_factory=BlackboxConfig)
    attack: str = "fgsm"


@dataclass
class Ensemble3ModelsConfig(MainConfig):
    data: DataConfig = field(default_factory=lambda: DataConfig(batch_size=1))
    model: ModelConfig = field(
        default_factory=lambda: ModelConfig(
            use_source_crop=True,
            use_target_crop=True,
            backbone=["B16", "B32", "Laion"],
        )
    )


cs = ConfigStore.instance()
cs.store(name="config", node=MainConfig)
cs.store(name="ensemble_3models", node=Ensemble3ModelsConfig)
"""

schema_path.write_text(patched_schema, encoding="utf-8")
print("Patched:", schema_path)

foattack_path = REPO_DIR / "FOAttack.py"
if foattack_path.exists():
    text = foattack_path.read_text(encoding="utf-8")
    text = text.replace(
        'os.environ["CUDA_VISIBLE_DEVICES"] = "2"',
        '# Kaggle patch: do not hard-code physical GPU 2; Hydra uses cuda:0'
    )
    foattack_path.write_text(text, encoding="utf-8")
    print("Patched:", foattack_path)

subprocess.run([str(VENV_PY), "-m", "py_compile", str(schema_path)], check=True)
print("Patch syntax check: OK")


In [ ]:
# Cell 6 — Locate the source and target image folders.

def count_images(path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sum(1 for p in path.rglob("*") if p.is_file() and p.suffix.lower() in exts)

def find_named_dirs(root, name):
    return sorted(
        [p for p in root.rglob(name) if p.is_dir()],
        key=lambda p: (-count_images(p), len(p.parts))
    )

clean_candidates = find_named_dirs(DATASET_ROOT, "bigscale")
target_candidates = find_named_dirs(DATASET_ROOT, "target_images")

print("bigscale candidates:")
for p in clean_candidates[:10]:
    print(" ", p, "images =", count_images(p))

print("\ntarget_images candidates:")
for p in target_candidates[:10]:
    print(" ", p, "images =", count_images(p))

if not clean_candidates:
    raise FileNotFoundError("Cannot find a directory named 'bigscale' in the dataset.")
if not target_candidates:
    raise FileNotFoundError("Cannot find a directory named 'target_images' in the dataset.")

CLEAN_DATA = clean_candidates[0]
TARGET_DATA = target_candidates[0]

def validate_imagefolder_root(path, label):
    direct_dirs = [p for p in path.iterdir() if p.is_dir()]
    if not direct_dirs:
        raise RuntimeError(
            f"{label}={path} has no class subdirectory. "
            "torchvision.datasets.ImageFolder requires at least one subfolder."
        )
    n = count_images(path)
    if n == 0:
        raise RuntimeError(f"No images found under {label}={path}")
    print(f"{label}: {path}")
    print("  class folders:", [p.name for p in direct_dirs[:10]])
    print("  image count:", n)

validate_imagefolder_root(CLEAN_DATA, "CLEAN_DATA")
validate_imagefolder_root(TARGET_DATA, "TARGET_DATA")


In [ ]:
# Cell 7 — Verify versions, CUDA, and the missing kmeans dependency.

import subprocess

check_code = r"""
import sys
from importlib import metadata
import torch
import torchvision
import transformers
import hydra
import omegaconf
import wandb
from kmeans_pytorch import kmeans

print("Python              :", sys.version.split()[0])
print("torch               :", torch.__version__)
print("torchvision         :", torchvision.__version__)
print("transformers        :", transformers.__version__)
print("hydra-core          :", metadata.version("hydra-core"))
print("omegaconf           :", omegaconf.__version__)
print("pytorch-lightning   :", metadata.version("pytorch-lightning"))
print("wandb               :", wandb.__version__)
print("kmeans-pytorch      :", metadata.version("kmeans-pytorch"))
print("CUDA available      :", torch.cuda.is_available())
print("CUDA runtime (torch):", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU accelerator in Kaggle.")

print("GPU                  :", torch.cuda.get_device_name(0))

x = torch.randn(32, 8, device="cuda")
ids, centers = kmeans(
    X=x,
    num_clusters=4,
    distance="euclidean",
    device=torch.device("cuda:0"),
    tqdm_flag=False,
)
print("kmeans GPU test      : OK", tuple(centers.shape))
"""

subprocess.run([str(VENV_PY), "-c", check_code], check=True)


In [ ]:
# Cell 8 — Write a Kaggle-specific Hydra config based on ensemble_3models.yaml.

CONFIG_PATH = REPO_DIR / "config" / "kaggle_foa.yaml"

def write_config(num_samples, steps):
    config_text = f"""
defaults:
  - _self_

data:
  batch_size: 1
  num_samples: {int(num_samples)}
  output: "{OUTPUT_DIR}"
  cle_data_path: "{CLEAN_DATA}"
  tgt_data_path: "{TARGET_DATA}"

optim:
  alpha: 1.0
  epsilon: 16
  steps: {int(steps)}

model:
  input_res: 224
  use_source_crop: true
  use_target_crop: true
  crop_scale: [0.5, 0.9]
  ensemble: true
  device: "cuda:0"
  backbone: ["B16", "B32", "Laion"]

wandb:
  project: "local_adversarial_attack"
  entity: null

blackbox:
  model_name: "gemini"
  batch_size: 1
  timeout: 30

attack: "fgsm"
""".strip() + "\n"
    CONFIG_PATH.write_text(config_text, encoding="utf-8")
    print(CONFIG_PATH.read_text())

write_config(SMOKE_NUM_SAMPLES, SMOKE_STEPS)


## Model download requirement

The attack code loads:

- `openai/clip-vit-base-patch16`
- `openai/clip-vit-base-patch32`
- `laion/CLIP-ViT-G-14-laion2B-s12B-b42K`

If they are not already cached/attached, turn **Kaggle Internet ON** before the first attack run.

The notebook sets `WANDB_MODE=disabled`, so W&B login is not required for adversarial-image generation.


In [ ]:
# Cell 9 — End-to-end smoke test.
# 1 image pair, 2 iterations, but still loads all three real surrogate models.

import os
import subprocess

write_config(SMOKE_NUM_SAMPLES, SMOKE_STEPS)

env = os.environ.copy()
env["WANDB_MODE"] = "disabled"
env["WANDB_SILENT"] = "true"
env["HF_HOME"] = str(HF_HOME)
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    str(VENV_PY),
    str(REPO_DIR / "generate_adversarial_samples_foa_attack.py"),
    "--config-name", "kaggle_foa",
]

print("Running:", " ".join(cmd))
print("Working directory:", REPO_DIR)

subprocess.run(cmd, cwd=str(REPO_DIR), env=env, check=True)

print("\nSMOKE TEST PASSED")
print("Output:", OUTPUT_DIR)


In [ ]:
# Cell 10 — Full attack using the repo/paper default config.
# Set RUN_FULL_ATTACK=True only after Cell 9 passes.

RUN_FULL_ATTACK = False

if RUN_FULL_ATTACK:
    import os
    import shutil
    import subprocess

    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

    write_config(PAPER_NUM_SAMPLES, PAPER_STEPS)

    env = os.environ.copy()
    env["WANDB_MODE"] = "disabled"
    env["WANDB_SILENT"] = "true"
    env["HF_HOME"] = str(HF_HOME)
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["PYTHONUNBUFFERED"] = "1"

    cmd = [
        str(VENV_PY),
        str(REPO_DIR / "generate_adversarial_samples_foa_attack.py"),
        "--config-name", "kaggle_foa",
    ]

    print("Starting full FOA-Attack...")
    subprocess.run(cmd, cwd=str(REPO_DIR), env=env, check=True)
    print("Full attack completed:", OUTPUT_DIR)
else:
    print("Full attack disabled.")
    print("Set RUN_FULL_ATTACK=True and rerun this cell after smoke test succeeds.")


In [ ]:
# Cell 11 — Inspect and zip outputs.

import shutil

if not OUTPUT_DIR.exists():
    print("No output yet:", OUTPUT_DIR)
else:
    images = sorted(
        p for p in OUTPUT_DIR.rglob("*")
        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    )

    print("Generated images:", len(images))
    for p in images[:20]:
        print(p)

    zip_path = shutil.make_archive(
        "/kaggle/working/foa_attack_output",
        "zip",
        root_dir=str(OUTPUT_DIR),
    )
    print("ZIP:", zip_path)


## Optional: `FOAttack.py`

`FOAttack.py` is a larger path than basic adversarial-image generation. It imports OpenAI, Anthropic and Google GenAI clients and reads API keys.

If you need that path later, install these repo-pinned extras in the venv:

```bash
/kaggle/working/foa_venv/bin/pip install   openai==1.63.0   anthropic==0.45.2   google-genai==1.2.0   tenacity==9.0.0
```

Store API keys in Kaggle Secrets rather than committing them to the notebook or dataset.
